# ALL QUANTUM — Run All 13 Models (Conjunctiva Image → Hb Prediction)

Conjunctiva/eye image + Excel → Haemoglobin (Hb) estimation using Quantum ML.
All model code loads from **this repo only** — no external downloads needed.

| # | Model | Folder | Publication |
|---|-------|--------|-------------|
| 1 | QSVM | 01_QSVM_QuantumEnhancedFeatureSpaces | Nature 567 (2019) — IBM |
| 2 | VQC | 02_VQC_VariationalQuantumClassifier | arXiv 1802.06002 (2018) — Google |
| 3 | QCNN | 03_QCNN_QuantumConvolutionalNeuralNet | Nature Physics 15 (2019) — Harvard |
| 4 | TorchQuantum VQA | 04_VQA_TorchQuantum_Framework | Nat. Rev. Physics 3 (2021) — MIT |
| 5 | Data Re-Uploading | 05_DataReUploading_UniversalClassifier | Quantum 4, 226 (2020) |
| 6 | Quantum Transfer Learning | 06_QuantumTransferLearning | Quantum 4, 340 (2020) — Xanadu |
| 7 | Quantum Kernel Methods | 07_QuantumKernelMethods | PRX Quantum (2021) — Xanadu |
| 8 | Quantum Random Forest | 08_QuantumRandomForest | Springer QMI (2023) |
| 9 | QBoost | 09_QBoost_QuantumBoosting | ACML 2012 — Google/D-Wave |
| 10 | CV-QNN | 10_CVQNN_ContinuousVariableQNN | Phys Rev Research 1 (2019) — Xanadu |
| B1 | QSVM Medical | 11_BONUS_QSVM_MedicalClassifier | Bonus — medical template |
| B2 | QCNN Phases | 12_BONUS_QCNN_PhasesOfMatter | Bonus — Cong et al. exact impl. |
| B3 | Meta-Learning QNN | 13_BONUS_VQA_MetaLearning_QNN | Bonus — Verdon et al. 2019 |


## Section 1 — CONFIGURATION  (Edit only this cell)

In [ ]:
# ==============================================================
#         ALL VARIABLES — EDIT ONLY THIS BLOCK
# ==============================================================

# ── Task ──────────────────────────────────────────────────────
TASK = "regression"   # "regression" | "classification"

# ── Dataset ───────────────────────────────────────────────────
IMAGE_DIR  = "/kaggle/input/your-dataset/images"
EXCEL_PATH = "/kaggle/input/your-dataset/data.xlsx"
IMAGE_COL  = "Patient_ID"
HB_COL     = "Hemoglobin"

# ── Classification threshold ───────────────────────────────────
HB_THRESH = 12.0

# ── Hb range filter ───────────────────────────────────────────
HB_FILTER_MIN = 5.0
HB_FILTER_MAX = 18.0

# ── Quantum circuit settings ──────────────────────────────────
N_QUBITS  = 4
N_LAYERS  = 2

# ── Feature extraction backbone ───────────────────────────────
BACKBONE        = "resnet18"
FREEZE_BACKBONE = True

# ── Training ──────────────────────────────────────────────────
EPOCHS     = 30           # increased from 15 — gives all models room to converge
BATCH_SIZE = 16
LR         = 1e-3
VAL_SPLIT  = 0.2
TEST_SPLIT = 0.1
SEED       = 42
EARLY_STOP = 12           # increased from 5 — avoids premature stopping

# ── Regression normalisation ──────────────────────────────────
REG_CONFIG = {
    "normalize_hb" : True,
    "hb_min"       : 4.0,
    "hb_max"       : 20.0,
    "loss_fn"      : "huber",
    "huber_delta"  : 1.0,
}

# ── Classification settings ───────────────────────────────────
CLS_CONFIG = {
    "use_focal_loss"    : True,
    "focal_gamma"       : 2.0,
    "use_class_weights" : True,
    "threshold"         : 0.5,
}

# ── Which models to run ───────────────────────────────────────
RUN = {
    "QSVM"             : True,
    "VQC"              : True,
    "QCNN"             : True,
    "TorchQuantum_VQA" : True,
    "DataReUploading"  : True,
    "QuantumTransferLearning": True,
    "QuantumKernel"    : True,
    "QuantumRandForest": True,
    "QBoost"           : True,
    "CVQNN"            : True,
    "QSVM_Medical"     : True,
    "QCNN_Phases"      : True,
    "MetaLearning_QNN" : True,
}


## Step 0 — Clone Repo  *(run once on Kaggle / Colab)*
> On local: skip this cell if you already have the repo checked out.

In [ ]:
import os, sys, subprocess

REPO_URL  = "https://github.com/junaidmaqbool/QuantumHb.git"
CLONE_DIR = "/kaggle/working/QuantumHb"

if not os.path.isdir(CLONE_DIR):
    print("Cloning QuantumHb ...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CLONE_DIR], check=True)
    print(f"✓ Cloned to {CLONE_DIR}")
else:
    print("✓ Repo already present — pulling latest ...")
    subprocess.run(["git", "-C", CLONE_DIR, "pull", "--rebase"], check=True)

REPO_ROOT = CLONE_DIR
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)

print(f"REPO_ROOT : {REPO_ROOT}")
print(f"TASK      : {TASK}")
print(f"N_QUBITS  : {N_QUBITS}")
print(f"Models ON : {[k for k,v in RUN.items() if v]}")


## Section 2 — Install Dependencies

In [ ]:
import subprocess, sys

PKGS = [
    "pennylane",             # core quantum ML framework (all PL models)
    "pennylane-qiskit",      # PennyLane-Qiskit bridge
    "qiskit",                # Qiskit core
    "qiskit-machine-learning",  # QSVM, quantum kernels
    "torchvision",           # ResNet-18 backbone
    "timm",                  # image model zoo
    "pandas",
    "openpyxl",              # read Excel files
    "scikit-learn",
    "matplotlib",
    "tqdm",
    "einops",
    "scipy",
]

for pkg in PKGS:
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", pkg, "-q"],
        capture_output=True, text=True)
    status = "OK  " if r.returncode == 0 else "WARN"
    print(f"  {status} {pkg}")

print("\nDependencies done.")


## Section 3 — Imports & Dataset

In [ ]:
import os, sys, math, time, warnings, traceback
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, roc_auc_score,
                              mean_absolute_error, f1_score, mean_squared_error)
from tqdm import tqdm
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

torch.manual_seed(SEED)
np.random.seed(SEED)
# ── CUDA sanity check (guards against compute-capability mismatches) ──
DEVICE = "cpu"
if torch.cuda.is_available():
    try:
        _t = torch.zeros(1, device="cuda")
        _ = _t + _t          # trigger an actual kernel — catches arch mismatches
        del _t
        DEVICE = "cuda"
        print(f"  GPU : {torch.cuda.get_device_name(0)}")
    except Exception as _cuda_err:
        print(f"  ⚠ CUDA available but kernel mismatch → falling back to CPU")
        print(f"    ({_cuda_err})")
        torch.cuda.empty_cache()
print(f"Device  : {DEVICE}   PyTorch: {torch.__version__}")
print(f"Task    : {TASK}")
print(f"Qubits  : {N_QUBITS}   Layers: {N_LAYERS}")


---
## Section 3b — Dataset, Feature Extraction & Loaders

- Reads Excel → filters HB range → builds train/val/test split
- ResNet-18 backbone (frozen) extracts 512-dim features from each image
- Linear(512 → N_QUBITS) trains alongside the quantum circuit
- Gradient-based models receive `DataLoader` of `(features, label)` tensors
- SVM-based models receive plain NumPy arrays `(X_train, y_train)` etc.


In [ ]:
# ── Image transforms ──────────────────────────────────────────
_IMG_EXTS = [".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ""]

def _find_img(patient_id, img_dir):
    """Locate image file for a given Patient_ID."""
    for ext in _IMG_EXTS:
        p = os.path.join(img_dir, str(patient_id) + ext)
        if os.path.exists(p):
            return p
    return None

TRANSFORM_TRAIN = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
TRANSFORM_EVAL = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class HbDataset(Dataset):
    def __init__(self, records, img_dir, transform, task):
        self.records   = records         # list of (patient_id, hb_value)
        self.img_dir   = img_dir
        self.transform = transform
        self.task      = task

    def __len__(self): return len(self.records)

    def __getitem__(self, idx):
        pid, hb = self.records[idx]
        path    = _find_img(pid, self.img_dir)
        if path is None:
            img = Image.fromarray(np.zeros((224, 224, 3), dtype=np.uint8))
        else:
            img = Image.open(path).convert("RGB")
        img = self.transform(img)
        if self.task == "classification":
            label = torch.tensor(1.0 if hb >= HB_THRESH else 0.0)
        else:
            lo, hi = REG_CONFIG["hb_min"], REG_CONFIG["hb_max"]
            label  = torch.tensor((hb - lo) / (hi - lo), dtype=torch.float32)
        return img, label

# ── Load Excel ────────────────────────────────────────────────
print("Loading dataset...")
# ── Auto-detect column names & load Excel ────────────────────────────────────
print("Loading dataset...")

# Step 1: read the file, show all columns so user can verify
df_raw = pd.read_excel(EXCEL_PATH)
print(f"  Excel loaded: {df_raw.shape[0]} rows × {df_raw.shape[1]} cols")
print(f"  Columns found: {list(df_raw.columns)}")

# Step 2: resolve IMAGE_COL
def _resolve_col(df, user_col, keywords, role):
    if user_col in df.columns:
        return user_col
    # case-insensitive exact match
    lower_map = {c.lower().strip(): c for c in df.columns}
    if user_col.lower().strip() in lower_map:
        found = lower_map[user_col.lower().strip()]
        print(f"  ⚠ {role}: '{user_col}' → matched as '{found}' (case-insensitive)")
        return found
    # keyword search
    for kw in keywords:
        hits = [c for c in df.columns if kw.lower() in c.lower()]
        if hits:
            print(f"  ⚠ {role}: '{user_col}' not found → auto-detected '{hits[0]}' (contains '{kw}')")
            return hits[0]
    # last resort: list columns and raise
    raise KeyError(
        f"\n  ✗ Could not find {role} column.\n"
        f"    Configured: '{user_col}'\n"
        f"    Available : {list(df.columns)}\n"
        f"    → Set {role.split()[0].upper()}_COL in Section 1 to one of the above."
    )

img_col = _resolve_col(df_raw, IMAGE_COL,
                        ["id","patient","file","image","img","name","photo","pic","sample"],
                        "IMAGE_COL (patient/image ID)")
hb_col  = _resolve_col(df_raw, HB_COL,
                        ["hb","hemoglobin","haemoglobin","hgb","hbg","anemia","anaemia"],
                        "HB_COL (haemoglobin value)")
print(f"  Using → IMAGE_COL='{img_col}'   HB_COL='{hb_col}'")

# Step 3: clean and filter
df = df_raw[[img_col, hb_col]].copy()
df.columns = ["_img_id", "_hb"]
df["_hb"] = pd.to_numeric(df["_hb"], errors="coerce")
df = df.dropna(subset=["_hb"])
n_before = len(df)
df = df[(df["_hb"] >= HB_FILTER_MIN) & (df["_hb"] <= HB_FILTER_MAX)].reset_index(drop=True)
print(f"  After Hb filter [{HB_FILTER_MIN}–{HB_FILTER_MAX} g/dL]: {len(df)} rows "
      f"({n_before - len(df)} dropped)")
print(f"  Hb range : {df['_hb'].min():.1f} – {df['_hb'].max():.1f} g/dL")
if TASK == "classification":
    n_anemic = (df["_hb"] < HB_THRESH).sum()
    print(f"  Classes  : Anemic={n_anemic}  Normal={len(df)-n_anemic}  (thresh={HB_THRESH})")

records = list(zip(df["_img_id"].astype(str), df["_hb"].astype(float)))
hb_vals = df["_hb"].astype(float).values

# ── Train / val / test split ──────────────────────────────────
idx      = np.arange(len(records))
idx_tv, idx_test = train_test_split(idx, test_size=TEST_SPLIT, random_state=SEED)
idx_tr, idx_val  = train_test_split(idx_tv, test_size=VAL_SPLIT, random_state=SEED)

tr_recs  = [records[i] for i in idx_tr]
val_recs = [records[i] for i in idx_val]
te_recs  = [records[i] for i in idx_test]
print(f"  Split → train:{len(tr_recs)}  val:{len(val_recs)}  test:{len(te_recs)}")

train_ds = HbDataset(tr_recs,  IMAGE_DIR, TRANSFORM_TRAIN, TASK)
val_ds   = HbDataset(val_recs, IMAGE_DIR, TRANSFORM_EVAL,  TASK)
test_ds  = HbDataset(te_recs,  IMAGE_DIR, TRANSFORM_EVAL,  TASK)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=False)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=False)
print("DataLoaders ready.")


## Section 3c — ResNet-18 Feature Extractor + Quantum Input Reducer

In [ ]:
# ── ResNet-18 backbone (shared across all models) ────────────
_resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
_resnet.fc = nn.Identity()          # remove classification head → 512-dim output
_resnet = _resnet.to(DEVICE)
if FREEZE_BACKBONE:
    for p in _resnet.parameters():
        p.requires_grad_(False)
_resnet.eval()
print(f"ResNet-18 backbone ready  (frozen={FREEZE_BACKBONE})")

def extract_features_numpy(loader, desc="Extracting"):
    """
    Extract ResNet-18 512-dim features for an entire DataLoader.
    Returns (X: np.ndarray shape [N,512], y: np.ndarray shape [N])
    Used by SVM-based quantum models.
    """
    Xs, ys = [], []
    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc=desc, leave=False):
            feats = _resnet(imgs.to(DEVICE)).cpu().numpy()
            Xs.append(feats)
            ys.append(labels.numpy())
    X = np.concatenate(Xs); y = np.concatenate(ys)
    # PCA reduce 512 → N_QUBITS for quantum circuits
    return X, y

def make_pca_reduced(X_tr, X_val, X_te, n_components):
    """StandardScale + PCA to N_QUBITS for SVM-style quantum models."""
    sc  = StandardScaler()
    X_tr_s  = sc.fit_transform(X_tr)
    X_val_s = sc.transform(X_val)
    X_te_s  = sc.transform(X_te)
    pca = PCA(n_components=n_components, random_state=SEED)
    X_tr_p  = pca.fit_transform(X_tr_s)
    X_val_p = pca.transform(X_val_s)
    X_te_p  = pca.transform(X_te_s)
    print(f"  PCA variance retained: {pca.explained_variance_ratio_.sum()*100:.1f}%")
    return X_tr_p, X_val_p, X_te_p

print("Feature extraction helpers ready.")


## Section 4 — Training Utilities

In [ ]:
# ── Results collector ─────────────────────────────────────────
RESULTS        = []
LEARNING_CURVES = {}   # name → {"tr": [...], "vl": [...], "val_mae": [...]}
MODEL_PREDS     = {}   # name → {"y_true": ndarray, "y_pred": ndarray}  (Hb g/dL)

# ── Loss functions ────────────────────────────────────────────
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None):
        super().__init__()
        self.gamma = gamma; self.weight = weight
    def forward(self, logits, targets):
        bce  = F.binary_cross_entropy(logits, targets, reduction="none")
        pt   = torch.exp(-bce)
        loss = ((1 - pt) ** self.gamma) * bce
        return loss.mean()

def _get_loss_fn(y_train_np=None):
    if TASK == "classification":
        if CLS_CONFIG["use_focal_loss"]:
            return FocalLoss(gamma=CLS_CONFIG["focal_gamma"])
        return nn.BCELoss()
    else:
        fn = REG_CONFIG["loss_fn"]
        if fn == "huber": return nn.HuberLoss(delta=REG_CONFIG["huber_delta"])
        if fn == "mae":   return nn.L1Loss()
        return nn.MSELoss()

def _denorm_hb(val):
    lo, hi = REG_CONFIG["hb_min"], REG_CONFIG["hb_max"]
    return val * (hi - lo) + lo

def _metrics_cls(y_true, y_prob):
    y_pred = (y_prob >= CLS_CONFIG["threshold"]).astype(int)
    acc  = accuracy_score(y_true, y_pred)
    f1   = f1_score(y_true, y_pred, zero_division=0)
    try:   auc = roc_auc_score(y_true, y_prob)
    except: auc = float("nan")
    return acc, f1, auc

def _metrics_reg(y_true_norm, y_pred_norm):
    # Clamp predictions to valid normalised range [0,1] before denormalising.
    # Prevents models with unbounded outputs (expval in [-1,1] or learnable
    # scale) from inflating MAE with physically impossible Hb values.
    y_true = _denorm_hb(np.clip(np.array(y_true_norm), 0.0, 1.0))
    y_pred = _denorm_hb(np.clip(np.array(y_pred_norm), 0.0, 1.0))
    mae    = mean_absolute_error(y_true, y_pred)
    rmse   = mean_squared_error(y_true, y_pred) ** 0.5
    return mae, rmse

# ── QuantumNet wrapper ─────────────────────────────────────────
class QuantumNet(nn.Module):
    def __init__(self, quantum_model):
        super().__init__()
        self.backbone = _resnet
        self.reducer  = nn.Linear(512, N_QUBITS)
        self.bn       = nn.BatchNorm1d(N_QUBITS)
        self.qmodel   = quantum_model

    def forward(self, imgs):
        with torch.set_grad_enabled(not FREEZE_BACKBONE):
            feats = self.backbone(imgs)
        x = torch.tanh(self.bn(self.reducer(feats))) * math.pi
        return self.qmodel(x)

# ── Per-model 4-panel diagnostic figure ──────────────────────
def _plot_model_results(name, y_true_hb, y_pred_hb,
                        tr_losses=None, vl_losses=None):
    """
    4-panel figure printed immediately after each model finishes:
      [0] Train / Val loss curves (gradient models) or info text (SVM)
      [1] Predicted vs True Hb scatter + identity line + threshold
      [2] Residual (Pred−True) histogram with bias marker
      [3] Confusion matrix derived from Hb threshold (regression → clinical)
    """
    import copy
    from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

    y_true = np.array(y_true_hb, dtype=float)
    y_pred = np.array(y_pred_hb, dtype=float)
    resid  = y_pred - y_true
    mae    = np.mean(np.abs(resid))
    rmse   = np.sqrt(np.mean(resid ** 2))
    bias   = np.mean(resid)

    # Anemia labels from regression output at HB_THRESH
    tc = (y_true >= HB_THRESH).astype(int)
    pc = (y_pred >= HB_THRESH).astype(int)

    has_curves = tr_losses is not None and len(tr_losses) > 0

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(
        f"{name[:60]}\nMAE = {mae:.3f} g/dL  |  RMSE = {rmse:.3f}  |  "
        f"Bias = {bias:+.3f}  |  n = {len(y_true)}",
        fontsize=10, fontweight="bold", y=1.01,
    )

    # ── [0,0] Loss curves ──────────────────────────────────────
    ax = axes[0, 0]
    if has_curves:
        ep = range(1, len(tr_losses) + 1)
        ax.plot(ep, tr_losses, "b-o", ms=3, lw=1.5, label="Train loss")
        if vl_losses:
            ax.plot(ep, vl_losses, "r-s", ms=3, lw=1.5, label="Val loss")
        ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
        ax.set_title("Training & Validation Loss", fontsize=10, fontweight="bold")
        ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    else:
        ax.text(0.5, 0.5, "SVM / kernel model\n(no epoch-level curves)",
                ha="center", va="center", transform=ax.transAxes,
                fontsize=11, color="gray")
        ax.set_title("Loss Curves", fontsize=10, fontweight="bold")
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

    # ── [0,1] Predicted vs True scatter ───────────────────────
    ax = axes[0, 1]
    lo = min(y_true.min(), y_pred.min()) - 0.5
    hi = max(y_true.max(), y_pred.max()) + 0.5
    ax.scatter(y_true, y_pred, alpha=0.55, s=28, c="steelblue",
               edgecolors="k", linewidths=0.3, zorder=3)
    ax.plot([lo, hi], [lo, hi], "k--", lw=1.2, label="Identity (perfect)")
    ax.axhline(HB_THRESH, color="crimson", lw=0.9, ls=":",
               alpha=0.8, label=f"Thresh {HB_THRESH} g/dL")
    ax.axvline(HB_THRESH, color="crimson", lw=0.9, ls=":", alpha=0.8)
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
    ax.set_xlabel("True Hb (g/dL)"); ax.set_ylabel("Predicted Hb (g/dL)")
    ax.set_title("Predicted vs True Hb", fontsize=10, fontweight="bold")
    ax.legend(fontsize=7); ax.grid(True, alpha=0.25)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

    # ── [1,0] Residual histogram ───────────────────────────────
    ax = axes[1, 0]
    ax.hist(resid, bins=min(25, max(8, len(resid) // 4)),
            color="steelblue", edgecolor="white", alpha=0.8)
    ax.axvline(0,    color="black",  lw=1.3, ls="--", label="Zero bias")
    ax.axvline(bias, color="crimson", lw=1.3, ls="-",
               label=f"Mean bias = {bias:+.3f}")
    # ±1 g/dL tolerance band
    ax.axvline( 1.0, color="green", lw=0.9, ls=":", alpha=0.7, label="±1 g/dL")
    ax.axvline(-1.0, color="green", lw=0.9, ls=":", alpha=0.7)
    within1 = float((np.abs(resid) <= 1.0).mean() * 100)
    ax.set_xlabel("Residual (Pred − True) g/dL"); ax.set_ylabel("Count")
    ax.set_title(f"Residual Distribution  ({within1:.0f}% within ±1 g/dL)",
                 fontsize=10, fontweight="bold")
    ax.legend(fontsize=7); ax.grid(True, alpha=0.3)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

    # ── [1,1] Confusion matrix @ HB_THRESH ────────────────────
    ax = axes[1, 1]
    try:
        labels_uniq = sorted(set(tc) | set(pc))
        cm = confusion_matrix(tc, pc, labels=labels_uniq)
        disp_labels = []
        for l in labels_uniq:
            disp_labels.append("Anemic" if l == 0 else "Normal")
        disp = ConfusionMatrixDisplay(cm, display_labels=disp_labels)
        disp.plot(ax=ax, colorbar=False, cmap="Blues")
        if cm.size == 4:
            tn, fp, fn, tp = cm.ravel()
            sens = tp / max(1, tp + fn)
            spec = tn / max(1, tn + fp)
            f1_an = 2*tp / max(1, 2*tp + fp + fn)
            ax.set_xlabel(
                f"Predicted  |  Sens={sens:.2f}  Spec={spec:.2f}  F1={f1_an:.2f}",
                fontsize=8)
        ax.set_title(f"Confusion Matrix @ {HB_THRESH} g/dL",
                     fontsize=10, fontweight="bold")
    except Exception as _cm_err:
        ax.text(0.5, 0.5, f"CM error:\n{_cm_err}", ha="center", va="center",
                transform=ax.transAxes, fontsize=9)

    plt.tight_layout()
    safe = name[:35].replace(" ", "_").replace("/", "_").replace("—", "-")
    out  = os.path.join(REPO_ROOT, f"per_model_{safe}.png")
    plt.savefig(out, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"  📊 Saved → per_model_{safe}.png")

# ── Gradient-based trainer ────────────────────────────────────
def run_gradient_model(name, quantum_model):
    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")
    t0 = time.time()
    rec = {"name": name, "status": "failed", "params": 0, "time_s": 0,
           "model_type": "gradient"}

    try:
        net = QuantumNet(quantum_model).to(DEVICE)
        params = sum(p.numel() for p in net.parameters() if p.requires_grad)
        rec["params"] = params
        print(f"  Trainable params: {params:,}")

        opt      = torch.optim.Adam(
            filter(lambda p: p.requires_grad, net.parameters()), lr=LR)
        sched    = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
        loss_fn  = _get_loss_fn()
        best_val = float("inf"); patience = 0
        tr_losses, vl_losses, val_metrics_hist = [], [], []

        for epoch in range(1, EPOCHS + 1):
            # ── Train ──
            net.train()
            ep_loss = 0.0
            for imgs, labels in train_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                opt.zero_grad()
                out  = net(imgs)
                loss = loss_fn(out, labels)
                loss.backward()
                nn.utils.clip_grad_norm_(net.parameters(), 1.0)
                opt.step()
                ep_loss += loss.item()
            sched.step()
            ep_loss /= max(1, len(train_loader))

            # ── Validate ──
            net.eval()
            vl_loss = 0.0
            y_true_v, y_pred_v = [], []
            with torch.no_grad():
                for imgs, labels in val_loader:
                    imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                    out = net(imgs)
                    vl_loss += loss_fn(out, labels).item()
                    y_true_v.extend(labels.cpu().numpy())
                    y_pred_v.extend(out.cpu().numpy())
            vl_loss /= max(1, len(val_loader))
            tr_losses.append(ep_loss); vl_losses.append(vl_loss)

            if TASK == "regression":
                mae, rmse = _metrics_reg(y_true_v, y_pred_v)
                val_metrics_hist.append(mae)
                print(f"  Epoch {epoch:02d}/{EPOCHS}  tr={ep_loss:.4f}  "
                      f"vl={vl_loss:.4f}  val_MAE={mae:.3f} g/dL")
            else:
                acc, f1, auc = _metrics_cls(
                    np.array(y_true_v), np.array(y_pred_v))
                val_metrics_hist.append(auc)
                print(f"  Epoch {epoch:02d}/{EPOCHS}  tr={ep_loss:.4f}  "
                      f"vl={vl_loss:.4f}  acc={acc:.3f}  f1={f1:.3f}")

            if vl_loss < best_val:
                best_val = vl_loss; patience = 0
                best_state = copy.deepcopy(net.state_dict())
            else:
                patience += 1
            if EARLY_STOP > 0 and patience >= EARLY_STOP:
                print(f"  Early stop at epoch {epoch}"); break

        # Restore best weights
        if 'best_state' in dir():
            net.load_state_dict(best_state)

        # ── Test ──
        net.eval()
        y_true_t, y_pred_t = [], []
        with torch.no_grad():
            for imgs, labels in test_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                out = net(imgs)
                y_true_t.extend(labels.cpu().numpy())
                y_pred_t.extend(out.cpu().numpy())

        rec["status"]  = "ok"
        rec["time_s"]  = round(time.time() - t0, 1)
        rec["tr_loss"] = round(tr_losses[-1], 4)
        rec["vl_loss"] = round(vl_losses[-1], 4)

        if TASK == "regression":
            mae, rmse = _metrics_reg(y_true_t, y_pred_t)
            rec["test_mae"]  = round(mae, 3)
            rec["test_rmse"] = round(rmse, 3)
            val_mae_v, _ = _metrics_reg(
                list(zip(*[(yt, yp) for yt, yp in zip(y_true_v, y_pred_v)]))[0],
                list(zip(*[(yt, yp) for yt, yp in zip(y_true_v, y_pred_v)]))[1]) \
                if y_true_v else (float("nan"), float("nan"))
            rec["val_mae"] = round(val_metrics_hist[-1] if val_metrics_hist else float("nan"), 3)
            print(f"  ✅ TEST  MAE={mae:.3f} g/dL  RMSE={rmse:.3f}  ({rec['time_s']}s)")
        else:
            acc, f1, auc = _metrics_cls(
                np.array(y_true_t), np.array(y_pred_t))
            rec["test_acc"] = round(acc, 3)
            rec["test_f1"]  = round(f1, 3)
            rec["test_auc"] = round(float(auc), 3)
            rec["val_auc"]  = round(val_metrics_hist[-1] if val_metrics_hist else float("nan"), 3)
            print(f"  ✅ TEST  acc={acc:.3f}  f1={f1:.3f}  auc={auc:.3f}  ({rec['time_s']}s)")

        # ── Store learning curves + per-model diagnostic plot ──
        LEARNING_CURVES[name] = {
            "tr": tr_losses, "vl": vl_losses, "val_metrics": val_metrics_hist
        }
        if TASK == "regression":
            _y_true_hb = _denorm_hb(np.clip(np.array(y_true_t), 0.0, 1.0))
            _y_pred_hb = _denorm_hb(np.clip(np.array(y_pred_t), 0.0, 1.0))
            MODEL_PREDS[name] = {"y_true": _y_true_hb, "y_pred": _y_pred_hb}
            _plot_model_results(name, _y_true_hb, _y_pred_hb, tr_losses, vl_losses)

    except Exception as e:
        rec["error"] = str(e)[:120]
        print(f"  ❌ FAILED: {e}")
        traceback.print_exc()

    RESULTS.append(rec)
    return rec

# ── SVM/kernel-based trainer ──────────────────────────────────
_svm_feats = {}

def run_svm_model(name, estimator):
    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")
    t0  = time.time()
    rec = {"name": name, "status": "failed", "params": "n/a",
           "time_s": 0, "model_type": "svm"}

    try:
        if "X_tr" not in _svm_feats:
            print("  Extracting ResNet features for SVM models...")
            X_tr_r, y_tr = extract_features_numpy(train_loader, "train")
            X_vl_r, y_vl = extract_features_numpy(val_loader,   "val")
            X_te_r, y_te = extract_features_numpy(test_loader,  "test")
            X_tr_p, X_vl_p, X_te_p = make_pca_reduced(
                X_tr_r, X_vl_r, X_te_r, n_components=N_QUBITS)
            _svm_feats.update({
                "X_tr": X_tr_p, "y_tr": y_tr,
                "X_vl": X_vl_p, "y_vl": y_vl,
                "X_te": X_te_p, "y_te": y_te,
            })
            print(f"  Features cached: {X_tr_p.shape}")

        X_tr = _svm_feats["X_tr"]; y_tr = _svm_feats["y_tr"]
        X_vl = _svm_feats["X_vl"]; y_vl = _svm_feats["y_vl"]
        X_te = _svm_feats["X_te"]; y_te = _svm_feats["y_te"]

        if TASK == "classification":
            y_tr = (y_tr >= 0.5).astype(int)
            y_vl = (y_vl >= 0.5).astype(int)
            y_te = (y_te >= 0.5).astype(int)

        print(f"  Fitting {name}...")
        estimator.fit(X_tr, y_tr)
        rec["params"] = "n/a"

        if TASK == "regression":
            y_pred_v = np.clip(estimator.predict(X_vl), 0.0, 1.0)
            y_pred_t = np.clip(estimator.predict(X_te), 0.0, 1.0)
            mae_v, _ = _metrics_reg(y_vl, y_pred_v)
            mae, rmse = _metrics_reg(y_te, y_pred_t)
            rec.update({"test_mae": round(mae,3), "test_rmse": round(rmse,3),
                        "val_mae": round(mae_v,3)})
            print(f"  val_MAE={mae_v:.3f}  TEST  MAE={mae:.3f} g/dL  RMSE={rmse:.3f}")
            # store denorm predictions for summary plots
            _y_true_hb = _denorm_hb(np.clip(np.array(y_te), 0.0, 1.0))
            _y_pred_hb = _denorm_hb(np.clip(np.array(y_pred_t), 0.0, 1.0))
            MODEL_PREDS[name] = {"y_true": _y_true_hb, "y_pred": _y_pred_hb}
            _plot_model_results(name, _y_true_hb, _y_pred_hb)
        else:
            y_prob_v = estimator.predict_proba(X_vl) if hasattr(estimator,"predict_proba") \
                       else estimator.predict(X_vl).astype(float)
            if hasattr(y_prob_v, "ndim") and y_prob_v.ndim == 2:
                y_prob_v = y_prob_v[:, 1]
            y_prob_t = estimator.predict_proba(X_te) if hasattr(estimator,"predict_proba") \
                       else estimator.predict(X_te).astype(float)
            if hasattr(y_prob_t, "ndim") and y_prob_t.ndim == 2:
                y_prob_t = y_prob_t[:, 1]
            acc, f1, auc = _metrics_cls(y_te, y_prob_t)
            acc_v, f1_v, auc_v = _metrics_cls(y_vl, y_prob_v)
            rec.update({"test_acc": round(acc,3), "test_f1": round(f1,3),
                        "test_auc": round(float(auc),3),
                        "val_auc":  round(float(auc_v),3)})
            print(f"  ✅ TEST  acc={acc:.3f}  f1={f1:.3f}  auc={auc:.3f}")

        rec["status"] = "ok"
        rec["time_s"] = round(time.time() - t0, 1)
        print(f"  Time: {rec['time_s']}s")

    except Exception as e:
        rec["error"] = str(e)[:120]
        print(f"  ❌ FAILED: {e}")
        traceback.print_exc()

    RESULTS.append(rec)
    return rec

def _fail(name, e):
    print(f"  ❌ {name} FAILED: {e}")
    RESULTS.append({"name": name, "status": "failed", "error": str(e)[:120]})

def _add(folder):
    p = os.path.join(REPO_ROOT, folder)
    if p not in sys.path: sys.path.insert(0, p)

print("Training utilities ready.")


---
## Model QSVM — QSVM — Quantum-Enhanced Feature Spaces
*Havlíček et al., Nature 567 (2019) — IBM Quantum*

In [ ]:
if RUN["QSVM"]:
    _add("01_QSVM_QuantumEnhancedFeatureSpaces")
    try:
        import importlib.util as _ilu
        _spec = _ilu.spec_from_file_location(
            "_QSVM", os.path.join(REPO_ROOT, "01_QSVM_QuantumEnhancedFeatureSpaces", "eye_hb_model.py"))
        _mod = _ilu.module_from_spec(_spec); _spec.loader.exec_module(_mod)
        _, _estimator = _mod.build_model(n_qubits=N_QUBITS, task=TASK)
        run_svm_model("QSVM — Quantum-Enhanced Feature Spaces", _estimator)
    except Exception as _e:
        _fail("QSVM — Quantum-Enhanced Feature Spaces", _e)
else:
    print("QSVM — Quantum-Enhanced Feature Spaces skipped.")


---
## Model VQC — VQC — Variational Quantum Classifier
*Farhi & Neven, arXiv 1802.06002 (2018) — Google*

In [ ]:
if RUN["VQC"]:
    _add("02_VQC_VariationalQuantumClassifier")
    try:
        import importlib.util as _ilu
        _spec = _ilu.spec_from_file_location(
            "_VQC", os.path.join(REPO_ROOT, "02_VQC_VariationalQuantumClassifier", "eye_hb_model.py"))
        _mod = _ilu.module_from_spec(_spec); _spec.loader.exec_module(_mod)
        _model = _mod.build_model(
            n_features=N_QUBITS, n_qubits=N_QUBITS,
            n_layers=N_LAYERS, task=TASK)
        run_gradient_model("VQC — Variational Quantum Classifier", _model)
    except Exception as _e:
        _fail("VQC — Variational Quantum Classifier", _e)
else:
    print("VQC — Variational Quantum Classifier skipped.")


---
## Model QCNN — QCNN — Quantum Convolutional Neural Network
*Cong, Choi & Lukin, Nature Physics 15 (2019) — Harvard*

In [ ]:
if RUN["QCNN"]:
    _add("03_QCNN_QuantumConvolutionalNeuralNet")
    try:
        import importlib.util as _ilu
        _spec = _ilu.spec_from_file_location(
            "_QCNN", os.path.join(REPO_ROOT, "03_QCNN_QuantumConvolutionalNeuralNet", "eye_hb_model.py"))
        _mod = _ilu.module_from_spec(_spec); _spec.loader.exec_module(_mod)
        _model = _mod.build_model(
            n_features=N_QUBITS, n_qubits=N_QUBITS,
            n_layers=N_LAYERS, task=TASK)
        run_gradient_model("QCNN — Quantum Convolutional Neural Network", _model)
    except Exception as _e:
        _fail("QCNN — Quantum Convolutional Neural Network", _e)
else:
    print("QCNN — Quantum Convolutional Neural Network skipped.")


---
## Model TorchQuantum_VQA — TorchQuantum VQA — Variational Quantum Algorithm
*Cerezo et al., Nature Reviews Physics 3 (2021) — MIT / Los Alamos*

In [ ]:
if RUN["TorchQuantum_VQA"]:
    _add("04_VQA_TorchQuantum_Framework")
    try:
        import importlib.util as _ilu
        _spec = _ilu.spec_from_file_location(
            "_TorchQuantum_VQA", os.path.join(REPO_ROOT, "04_VQA_TorchQuantum_Framework", "eye_hb_model.py"))
        _mod = _ilu.module_from_spec(_spec); _spec.loader.exec_module(_mod)
        _model = _mod.build_model(
            n_features=N_QUBITS, n_qubits=N_QUBITS,
            n_layers=N_LAYERS, task=TASK)
        run_gradient_model("TorchQuantum VQA — Variational Quantum Algorithm", _model)
    except Exception as _e:
        _fail("TorchQuantum VQA — Variational Quantum Algorithm", _e)
else:
    print("TorchQuantum VQA — Variational Quantum Algorithm skipped.")


---
## Model DataReUploading — Data Re-Uploading — Universal Quantum Classifier
*Pérez-Salinas et al., Quantum 4, 226 (2020) — U. Barcelona*

In [ ]:
if RUN["DataReUploading"]:
    _add("05_DataReUploading_UniversalClassifier")
    try:
        import importlib.util as _ilu
        _spec = _ilu.spec_from_file_location(
            "_DataReUploading", os.path.join(REPO_ROOT, "05_DataReUploading_UniversalClassifier", "eye_hb_model.py"))
        _mod = _ilu.module_from_spec(_spec); _spec.loader.exec_module(_mod)
        _model = _mod.build_model(
            n_features=N_QUBITS, n_qubits=N_QUBITS,
            n_layers=N_LAYERS, task=TASK)
        run_gradient_model("Data Re-Uploading — Universal Quantum Classifier", _model)
    except Exception as _e:
        _fail("Data Re-Uploading — Universal Quantum Classifier", _e)
else:
    print("Data Re-Uploading — Universal Quantum Classifier skipped.")


---
## Model QuantumTransferLearning — Quantum Transfer Learning — Dressed Quantum Circuit
*Mari et al., Quantum 4, 340 (2020) — Xanadu AI*

In [ ]:
if RUN["QuantumTransferLearning"]:
    _add("06_QuantumTransferLearning")
    try:
        import importlib.util as _ilu
        _spec = _ilu.spec_from_file_location(
            "_QuantumTransferLearning", os.path.join(REPO_ROOT, "06_QuantumTransferLearning", "eye_hb_model.py"))
        _mod = _ilu.module_from_spec(_spec); _spec.loader.exec_module(_mod)
        _model = _mod.build_model(
            n_features=N_QUBITS, n_qubits=N_QUBITS,
            n_layers=N_LAYERS, task=TASK)
        run_gradient_model("Quantum Transfer Learning — Dressed Quantum Circuit", _model)
    except Exception as _e:
        _fail("Quantum Transfer Learning — Dressed Quantum Circuit", _e)
else:
    print("Quantum Transfer Learning — Dressed Quantum Circuit skipped.")


---
## Model QuantumKernel — Quantum Kernel Methods — Trainable Quantum Kernel
*Schuld, PRX Quantum (2021) — Xanadu AI*

In [ ]:
if RUN["QuantumKernel"]:
    _add("07_QuantumKernelMethods")
    try:
        import importlib.util as _ilu
        _spec = _ilu.spec_from_file_location(
            "_QuantumKernel", os.path.join(REPO_ROOT, "07_QuantumKernelMethods", "eye_hb_model.py"))
        _mod = _ilu.module_from_spec(_spec); _spec.loader.exec_module(_mod)
        _, _estimator = _mod.build_model(n_qubits=N_QUBITS, task=TASK)
        run_svm_model("Quantum Kernel Methods — Trainable Quantum Kernel", _estimator)
    except Exception as _e:
        _fail("Quantum Kernel Methods — Trainable Quantum Kernel", _e)
else:
    print("Quantum Kernel Methods — Trainable Quantum Kernel skipped.")


---
## Model QuantumRandForest — Quantum Random Forest — Kernel-Based QRF
*Srikumar et al., Quantum Machine Intelligence, Springer (2023)*

In [ ]:
if RUN["QuantumRandForest"]:
    _add("08_QuantumRandomForest")
    try:
        import importlib.util as _ilu
        _spec = _ilu.spec_from_file_location(
            "_QuantumRandForest", os.path.join(REPO_ROOT, "08_QuantumRandomForest", "eye_hb_model.py"))
        _mod = _ilu.module_from_spec(_spec); _spec.loader.exec_module(_mod)
        _, _estimator = _mod.build_model(n_qubits=N_QUBITS, task=TASK)
        run_svm_model("Quantum Random Forest — Kernel-Based QRF", _estimator)
    except Exception as _e:
        _fail("Quantum Random Forest — Kernel-Based QRF", _e)
else:
    print("Quantum Random Forest — Kernel-Based QRF skipped.")


---
## Model QBoost — QBoost — Quantum Adiabatic Boosting
*Neven, Denchev, Rose & Macready, ACML 2012 — Google / D-Wave*

In [ ]:
if RUN["QBoost"]:
    _add("09_QBoost_QuantumBoosting")
    try:
        import importlib.util as _ilu
        _spec = _ilu.spec_from_file_location(
            "_QBoost", os.path.join(REPO_ROOT, "09_QBoost_QuantumBoosting", "eye_hb_model.py"))
        _mod = _ilu.module_from_spec(_spec); _spec.loader.exec_module(_mod)
        _, _estimator = _mod.build_model(n_qubits=N_QUBITS, task=TASK)
        run_svm_model("QBoost — Quantum Adiabatic Boosting", _estimator)
    except Exception as _e:
        _fail("QBoost — Quantum Adiabatic Boosting", _e)
else:
    print("QBoost — Quantum Adiabatic Boosting skipped.")


---
## Model CVQNN — CV-QNN — Continuous-Variable Quantum Neural Network
*Killoran et al., Physical Review Research 1 (2019) — Xanadu AI*

In [ ]:
if RUN["CVQNN"]:
    _add("10_CVQNN_ContinuousVariableQNN")
    try:
        import importlib.util as _ilu
        _spec = _ilu.spec_from_file_location(
            "_CVQNN", os.path.join(REPO_ROOT, "10_CVQNN_ContinuousVariableQNN", "eye_hb_model.py"))
        _mod = _ilu.module_from_spec(_spec); _spec.loader.exec_module(_mod)
        _model = _mod.build_model(
            n_features=N_QUBITS, n_qubits=N_QUBITS,
            n_layers=N_LAYERS, task=TASK)
        run_gradient_model("CV-QNN — Continuous-Variable Quantum Neural Network", _model)
    except Exception as _e:
        _fail("CV-QNN — Continuous-Variable Quantum Neural Network", _e)
else:
    print("CV-QNN — Continuous-Variable Quantum Neural Network skipped.")


---
## Model QSVM_Medical — BONUS — QSVM Medical Classifier (PauliFeatureMap)
*GlazeDonuts/QSVM — medical data optimised template*

In [ ]:
if RUN["QSVM_Medical"]:
    _add("11_BONUS_QSVM_MedicalClassifier")
    try:
        import importlib.util as _ilu
        _spec = _ilu.spec_from_file_location(
            "_QSVM_Medical", os.path.join(REPO_ROOT, "11_BONUS_QSVM_MedicalClassifier", "eye_hb_model.py"))
        _mod = _ilu.module_from_spec(_spec); _spec.loader.exec_module(_mod)
        _, _estimator = _mod.build_model(n_qubits=N_QUBITS, task=TASK)
        run_svm_model("BONUS — QSVM Medical Classifier (PauliFeatureMap)", _estimator)
    except Exception as _e:
        _fail("BONUS — QSVM Medical Classifier (PauliFeatureMap)", _e)
else:
    print("BONUS — QSVM Medical Classifier (PauliFeatureMap) skipped.")


---
## Model QCNN_Phases — BONUS — QCNN Phases of Matter (Cong et al. exact impl.)
*Jaybsoni/Quantum-Convolutional-Neural-Networks*

In [ ]:
if RUN["QCNN_Phases"]:
    _add("12_BONUS_QCNN_PhasesOfMatter")
    try:
        import importlib.util as _ilu
        _spec = _ilu.spec_from_file_location(
            "_QCNN_Phases", os.path.join(REPO_ROOT, "12_BONUS_QCNN_PhasesOfMatter", "eye_hb_model.py"))
        _mod = _ilu.module_from_spec(_spec); _spec.loader.exec_module(_mod)
        _model = _mod.build_model(
            n_features=N_QUBITS, n_qubits=N_QUBITS,
            n_layers=N_LAYERS, task=TASK)
        run_gradient_model("BONUS — QCNN Phases of Matter (Cong et al. exact impl.)", _model)
    except Exception as _e:
        _fail("BONUS — QCNN Phases of Matter (Cong et al. exact impl.)", _e)
else:
    print("BONUS — QCNN Phases of Matter (Cong et al. exact impl.) skipped.")


---
## Model MetaLearning_QNN — BONUS — VQA Meta-Learning QNN (LSTM warm-start)
*Verdon et al., 'Learning to learn with QNNs' (2019)*

In [ ]:
if RUN["MetaLearning_QNN"]:
    _add("13_BONUS_VQA_MetaLearning_QNN")
    try:
        import importlib.util as _ilu
        _spec = _ilu.spec_from_file_location(
            "_MetaLearning_QNN", os.path.join(REPO_ROOT, "13_BONUS_VQA_MetaLearning_QNN", "eye_hb_model.py"))
        _mod = _ilu.module_from_spec(_spec); _spec.loader.exec_module(_mod)
        _model = _mod.build_model(
            n_features=N_QUBITS, n_qubits=N_QUBITS,
            n_layers=N_LAYERS, task=TASK)
        run_gradient_model("BONUS — VQA Meta-Learning QNN (LSTM warm-start)", _model)
    except Exception as _e:
        _fail("BONUS — VQA Meta-Learning QNN (LSTM warm-start)", _e)
else:
    print("BONUS — VQA Meta-Learning QNN (LSTM warm-start) skipped.")


---
## Results Summary

In [ ]:
if RESULTS:
    df_res = pd.DataFrame(RESULTS)

    base = ["name", "status", "params", "time_s", "model_type"]
    if TASK == "regression":
        extra = ["test_mae", "test_rmse", "val_mae", "tr_loss", "vl_loss"]
    else:
        extra = ["test_acc", "test_f1", "test_auc", "val_auc", "tr_loss", "vl_loss"]

    cols    = [c for c in base + extra if c in df_res.columns]
    df_show = df_res[cols].copy()

    # ── Format columns for display ─────────────────────────────
    if "params" in df_show:
        df_show["params"] = df_show["params"].apply(
            lambda x: f"{int(x)/1e3:.1f}K" if pd.notna(x) and str(x).isdigit() and int(x) > 0
                      else ("n/a" if x in [0,"n/a",None] else str(x)))
    if "time_s" in df_show:
        df_show["time_s"] = df_show["time_s"].apply(
            lambda x: f"{x:.0f}s" if pd.notna(x) and str(x).replace(".","").isdigit() else "n/a")
    # Replace NaN in loss columns with "—" for SVM models
    for col in ["tr_loss","vl_loss"]:
        if col in df_show:
            df_show[col] = df_show[col].apply(
                lambda x: f"{x:.4f}" if pd.notna(x) and x == x else "—  (SVM)")

    print("\n" + "="*78)
    print(f"  ALL QUANTUM — RESULTS SUMMARY   (Task: {TASK.upper()})  [EPOCHS={EPOCHS}  ES={EARLY_STOP}]")
    print("="*78)
    print(df_show.to_string(index=False))

    # ── Summary stats ─────────────────────────────────────────
    ok = df_res[df_res["status"] == "ok"]
    if not ok.empty and TASK == "regression":
        best = ok.loc[ok["test_mae"].idxmin()]
        print(f"\n  🏆  Best single model : {best['name']}")
        print(f"         Test MAE = {best['test_mae']:.3f} g/dL  | RMSE = {best.get('test_rmse', float('nan')):.3f}")
        print(f"         Mean MAE across all models = {ok['test_mae'].mean():.3f} g/dL")
else:
    print("No results collected — all models skipped or failed.")


---
## Section 5 — Comprehensive Comparison Charts

In [ ]:
# ══════════════════════════════════════════════════════════════
#   COMPREHENSIVE COMPARISON CHARTS
# ══════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np, pandas as pd
import warnings; warnings.filterwarnings("ignore")

ok = pd.DataFrame([r for r in RESULTS if r.get("status") == "ok"])
if ok.empty:
    print("No successful models to plot."); 
else:
    metric  = "test_mae"  if TASK == "regression" else "test_auc"
    xlabel  = "Test MAE (g/dL)" if TASK == "regression" else "Test AUC"
    ascending = TASK == "regression"

    ok = ok.sort_values(metric, ascending=ascending).reset_index(drop=True)
    names  = ok["name"].tolist()
    scores = ok[metric].tolist()
    colors_plasma = plt.cm.plasma(np.linspace(0.15, 0.85, len(ok)))

    fig = plt.figure(figsize=(22, 18))
    gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.38)

    # ── 1. Horizontal bar chart (main ranking) ──────────────────
    ax1 = fig.add_subplot(gs[0, :2])
    bars = ax1.barh(names, scores, color=colors_plasma, edgecolor="white", linewidth=0.5)
    bars[0].set_edgecolor("gold"); bars[0].set_linewidth(2.5)
    for bar, val in zip(bars, scores):
        ax1.text(val + 0.003, bar.get_y() + bar.get_height()/2,
                 f"{val:.3f}", va="center", fontsize=8.5, fontweight="bold")
    ax1.set_xlabel(xlabel, fontsize=10)
    ax1.set_title("Model Ranking — Test " + xlabel, fontsize=11, fontweight="bold")
    ax1.grid(axis="x", alpha=0.3, linestyle="--")
    ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)

    # ── 2. Model-type breakdown (gradient vs SVM) ────────────────
    ax2 = fig.add_subplot(gs[0, 2])
    type_groups = {}
    for _, row in ok.iterrows():
        mt = row.get("model_type", "gradient")
        type_groups.setdefault(mt, []).append(row[metric])
    type_labels = list(type_groups.keys())
    type_vals   = [np.mean(v) for v in type_groups.values()]
    type_stds   = [np.std(v)  for v in type_groups.values()]
    bar_colors  = ["#4C72B0", "#DD8452"][:len(type_labels)]
    ax2.bar(type_labels, type_vals, yerr=type_stds, capsize=6,
            color=bar_colors, edgecolor="white")
    ax2.set_ylabel(xlabel, fontsize=9)
    ax2.set_title("Gradient vs SVM
Mean ± Std", fontsize=10, fontweight="bold")
    ax2.grid(axis="y", alpha=0.3, linestyle="--")
    ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)

    # ── 3. Test MAE vs RMSE scatter (regression only) ────────────
    ax3 = fig.add_subplot(gs[1, 0])
    if TASK == "regression" and "test_rmse" in ok.columns:
        sc = ax3.scatter(ok["test_mae"], ok["test_rmse"],
                         s=90, c=colors_plasma, edgecolors="k", linewidths=0.4, zorder=3)
        for _, row in ok.iterrows():
            ax3.annotate(row["name"][:12], (row["test_mae"], row["test_rmse"]),
                         fontsize=6.5, xytext=(4, 2), textcoords="offset points")
        # y=x reference line
        mn, mx = ok["test_mae"].min() * 0.95, ok["test_mae"].max() * 1.05
        ax3.plot([mn, mx], [mn, mx], "k--", alpha=0.4, linewidth=1, label="MAE = RMSE")
        ax3.set_xlabel("Test MAE (g/dL)", fontsize=9)
        ax3.set_ylabel("Test RMSE (g/dL)", fontsize=9)
        ax3.set_title("MAE vs RMSE
(below diagonal = low variance errors)", fontsize=9, fontweight="bold")
        ax3.legend(fontsize=7); ax3.grid(True, alpha=0.3, linestyle="--")
    else:
        ax3.text(0.5, 0.5, "N/A for classification", ha="center", va="center",
                 transform=ax3.transAxes, fontsize=11)
    ax3.spines["top"].set_visible(False); ax3.spines["right"].set_visible(False)

    # ── 4. Training time vs performance ─────────────────────────
    ax4 = fig.add_subplot(gs[1, 1])
    if "time_s" in ok.columns:
        time_ok = ok.dropna(subset=["time_s"]).copy()
        time_ok["time_s"] = pd.to_numeric(time_ok["time_s"], errors="coerce")
        time_ok = time_ok.dropna(subset=["time_s"])
        if len(time_ok) > 0:
            sc4 = ax4.scatter(time_ok["time_s"], time_ok[metric],
                              s=80, c=plt.cm.viridis(np.linspace(0.2, 0.8, len(time_ok))),
                              edgecolors="k", linewidths=0.4, zorder=3)
            for _, row in time_ok.iterrows():
                ax4.annotate(row["name"][:11], (row["time_s"], row[metric]),
                             fontsize=6, xytext=(3, 2), textcoords="offset points")
    ax4.set_xlabel("Training time (s)", fontsize=9)
    ax4.set_ylabel(xlabel, fontsize=9)
    ax4.set_title("Training Time vs Performance
(Pareto frontier top-left)", fontsize=9, fontweight="bold")
    ax4.grid(True, alpha=0.3, linestyle="--")
    ax4.spines["top"].set_visible(False); ax4.spines["right"].set_visible(False)

    # ── 5. Overlay of all learning curves ──────────────────────
    ax5 = fig.add_subplot(gs[1, 2])
    cmap5 = plt.cm.tab20(np.linspace(0, 1, len(LEARNING_CURVES)))
    for ci, (mname, lc) in enumerate(LEARNING_CURVES.items()):
        vl = lc.get("vl", [])
        if vl:
            ax5.plot(range(1, len(vl)+1), vl, linewidth=1.2,
                     alpha=0.75, label=mname[:15], color=cmap5[ci])
    ax5.set_xlabel("Epoch", fontsize=9); ax5.set_ylabel("Val Loss", fontsize=9)
    ax5.set_title("Val Loss Curves — All Gradient Models", fontsize=9, fontweight="bold")
    ax5.legend(fontsize=5.5, ncol=2, loc="upper right")
    ax5.grid(True, alpha=0.25, linestyle="--")
    ax5.spines["top"].set_visible(False); ax5.spines["right"].set_visible(False)

    # ── 6. Params vs Performance (gradient models only) ─────────
    ax6 = fig.add_subplot(gs[2, 0])
    grad_ok = ok[ok.get("model_type", pd.Series(["gradient"]*len(ok))) == "gradient"] \
              if "model_type" in ok.columns else ok
    params_ok = grad_ok.dropna(subset=["params"]).copy()
    try:
        params_ok["params_n"] = pd.to_numeric(params_ok["params"], errors="coerce")
        params_ok = params_ok.dropna(subset=["params_n"])
        if len(params_ok) > 1:
            ax6.scatter(params_ok["params_n"], params_ok[metric],
                        s=80, c=plt.cm.plasma(np.linspace(0.2,0.8,len(params_ok))),
                        edgecolors="k", linewidths=0.4, zorder=3)
            for _, row in params_ok.iterrows():
                ax6.annotate(row["name"][:12], (row["params_n"], row[metric]),
                             fontsize=6, xytext=(3, 2), textcoords="offset points")
            ax6.set_xlabel("Trainable Parameters", fontsize=9)
            ax6.set_ylabel(xlabel, fontsize=9)
            ax6.set_title("Model Size vs Performance", fontsize=9, fontweight="bold")
    except Exception:
        ax6.text(0.5, 0.5, "Params N/A", ha="center", va="center", transform=ax6.transAxes)
    ax6.grid(True, alpha=0.3, linestyle="--")
    ax6.spines["top"].set_visible(False); ax6.spines["right"].set_visible(False)

    # ── 7. Violin / box plot of val_metrics history ─────────────
    ax7 = fig.add_subplot(gs[2, 1:])
    lc_data, lc_names = [], []
    for mname, lc in LEARNING_CURVES.items():
        vm = lc.get("val_metrics", [])
        if vm:
            lc_data.append(vm)
            lc_names.append(mname[:14])
    if lc_data:
        parts = ax7.violinplot(lc_data, showmedians=True, showextrema=True)
        for pc in parts["bodies"]:
            pc.set_alpha(0.65)
        ax7.set_xticks(range(1, len(lc_names)+1))
        ax7.set_xticklabels(lc_names, rotation=40, ha="right", fontsize=7)
        ax7.set_ylabel(("Val MAE (g/dL)" if TASK=="regression" else "Val AUC"), fontsize=9)
        ax7.set_title("Val Metric Distribution per Epoch
(violin over all epochs)", fontsize=9, fontweight="bold")
        ax7.grid(axis="y", alpha=0.3, linestyle="--")
    ax7.spines["top"].set_visible(False); ax7.spines["right"].set_visible(False)

    fig.suptitle(f"QuantumHb — ALL MODELS Comparison Dashboard  |  {TASK.upper()}",
                 fontsize=14, fontweight="bold", y=1.005)
    plt.savefig(os.path.join(REPO_ROOT, "all_models_comparison.png"),
                dpi=130, bbox_inches="tight")
    plt.show()

    print("\n📊 Comparison dashboard saved → all_models_comparison.png")

# ══════════════════════════════════════════════════════════════
#   ENHANCED PAPER FIGURES  (Radar, Bland-Altman, CDF, Resources)
# ══════════════════════════════════════════════════════════════
import math

_ok   = pd.DataFrame([r for r in RESULTS if r.get("status") == "ok"])
_preds = MODEL_PREDS   # name → {"y_true": ..., "y_pred": ...}

if not _ok.empty and TASK == "regression" and _preds:

    _names_all = _ok["name"].tolist()
    _mae_all   = _ok["test_mae"].tolist()
    _rmse_all  = _ok.get("test_rmse", pd.Series([float("nan")]*len(_ok))).tolist()

    # ── Compute per-model anemia F1 from stored predictions ──
    from sklearn.metrics import f1_score as _f1_score
    _f1_all = []
    for nm in _names_all:
        if nm in _preds:
            yt = (_preds[nm]["y_true"] >= HB_THRESH).astype(int)
            yp = (_preds[nm]["y_pred"] >= HB_THRESH).astype(int)
            _f1_all.append(_f1_score(yt, yp, zero_division=0))
        else:
            _f1_all.append(float("nan"))

    # ─────────────────────────────────────────────────────────
    # FIG A — RADAR / SPIDER CHART
    # Metrics: MAE (inverted→ higher=better), RMSE (inv), Anemia-F1
    # ─────────────────────────────────────────────────────────
    _radar_categories = ["1/(1+MAE)", "1/(1+RMSE)", "Anemia F1"]
    N_ax = len(_radar_categories)
    angles = [n / float(N_ax) * 2 * math.pi for n in range(N_ax)]
    angles += angles[:1]

    fig_r, ax_r = plt.subplots(figsize=(9, 9),
                               subplot_kw=dict(polar=True))
    ax_r.set_theta_offset(math.pi / 2)
    ax_r.set_theta_direction(-1)
    ax_r.set_xticks(angles[:-1])
    ax_r.set_xticklabels(_radar_categories, fontsize=10)
    ax_r.set_ylim(0, 1)
    ax_r.yaxis.set_ticklabels([])
    ax_r.grid(color="gray", linestyle="--", linewidth=0.5, alpha=0.5)

    _colors_r = plt.cm.tab20(np.linspace(0, 1, len(_names_all)))
    for i, (nm, mae_v, rmse_v, f1_v) in enumerate(
            zip(_names_all, _mae_all, _rmse_all, _f1_all)):
        vals = [
            1 / (1 + (mae_v  if not math.isnan(mae_v)  else 10)),
            1 / (1 + (rmse_v if not math.isnan(rmse_v) else 10)),
            f1_v if not math.isnan(f1_v) else 0.0,
        ]
        vals += vals[:1]
        ax_r.plot(angles, vals, lw=1.5, color=_colors_r[i],
                  label=nm[:20], alpha=0.85)
        ax_r.fill(angles, vals, color=_colors_r[i], alpha=0.08)

    ax_r.legend(loc="upper right", bbox_to_anchor=(1.45, 1.15),
                fontsize=7, ncol=1)
    ax_r.set_title("Radar — MAE · RMSE · Anemia F1\n(outer = better)",
                   fontsize=12, fontweight="bold", pad=20)
    plt.tight_layout()
    plt.savefig(os.path.join(REPO_ROOT, "summary_radar.png"),
                dpi=130, bbox_inches="tight")
    plt.show()
    print("📊 Radar chart saved → summary_radar.png")

    # ─────────────────────────────────────────────────────────
    # FIG B — BLAND-ALTMAN PLOTS (one per model, 3-col grid)
    # ─────────────────────────────────────────────────────────
    _mods_ba = [(nm, _preds[nm]) for nm in _names_all if nm in _preds]
    if _mods_ba:
        ncols_ba = 3
        nrows_ba = math.ceil(len(_mods_ba) / ncols_ba)
        fig_ba, axes_ba = plt.subplots(nrows_ba, ncols_ba,
                                       figsize=(6 * ncols_ba, 4 * nrows_ba))
        axes_ba = np.array(axes_ba).flatten()

        for idx_ba, (nm, pr) in enumerate(_mods_ba):
            ax_ba = axes_ba[idx_ba]
            yt = pr["y_true"]; yp = pr["y_pred"]
            mean_val = (yt + yp) / 2
            diff_val = yp - yt
            md   = diff_val.mean()
            sd   = diff_val.std()
            loa_u = md + 1.96 * sd
            loa_l = md - 1.96 * sd
            ax_ba.scatter(mean_val, diff_val, alpha=0.45, s=20,
                          c="steelblue", edgecolors="k", linewidths=0.2, zorder=3)
            ax_ba.axhline(md,    color="red",   lw=1.3, ls="-",
                          label=f"Bias {md:+.2f}")
            ax_ba.axhline(loa_u, color="orange", lw=1.1, ls="--",
                          label=f"+1.96σ={loa_u:.2f}")
            ax_ba.axhline(loa_l, color="orange", lw=1.1, ls="--",
                          label=f"−1.96σ={loa_l:.2f}")
            ax_ba.axhline(0,     color="black",  lw=0.8, ls=":", alpha=0.5)
            ax_ba.set_xlabel("Mean of True & Pred (g/dL)", fontsize=8)
            ax_ba.set_ylabel("Pred − True (g/dL)", fontsize=8)
            ax_ba.set_title(nm[:28], fontsize=8, fontweight="bold")
            ax_ba.legend(fontsize=5.5, loc="upper right")
            ax_ba.grid(True, alpha=0.25)
            ax_ba.spines["top"].set_visible(False)
            ax_ba.spines["right"].set_visible(False)

        # hide unused axes
        for ax_ba in axes_ba[len(_mods_ba):]:
            ax_ba.set_visible(False)

        fig_ba.suptitle("Bland-Altman Agreement Plots — All Models",
                        fontsize=13, fontweight="bold", y=1.01)
        plt.tight_layout()
        plt.savefig(os.path.join(REPO_ROOT, "summary_bland_altman.png"),
                    dpi=130, bbox_inches="tight")
        plt.show()
        print("📊 Bland-Altman grid saved → summary_bland_altman.png")

    # ─────────────────────────────────────────────────────────
    # FIG C — CUMULATIVE ERROR CDF
    # % of test samples within ±δ g/dL, for δ in [0, 5]
    # ─────────────────────────────────────────────────────────
    _mods_cdf = [(nm, _preds[nm]) for nm in _names_all if nm in _preds]
    if _mods_cdf:
        delta_grid = np.linspace(0, 5, 200)
        fig_cdf, ax_cdf = plt.subplots(figsize=(11, 6))
        _colors_cdf = plt.cm.tab20(np.linspace(0, 1, len(_mods_cdf)))

        for idx_cdf, (nm, pr) in enumerate(_mods_cdf):
            abs_err = np.abs(pr["y_pred"] - pr["y_true"])
            cdf_vals = [(abs_err <= d).mean() * 100 for d in delta_grid]
            ax_cdf.plot(delta_grid, cdf_vals, lw=1.8,
                        color=_colors_cdf[idx_cdf],
                        label=nm[:22], alpha=0.85)

        # clinical tolerance reference lines
        for tol, col in [(1.0, "green"), (2.0, "orange")]:
            ax_cdf.axvline(tol, color=col, lw=1.0, ls="--", alpha=0.6,
                           label=f"±{tol} g/dL")

        ax_cdf.set_xlabel("Absolute Error Tolerance δ (g/dL)", fontsize=11)
        ax_cdf.set_ylabel("% Test Samples within δ", fontsize=11)
        ax_cdf.set_title("Cumulative Error CDF — All Models\n"
                         "(higher & further-left = better)",
                         fontsize=12, fontweight="bold")
        ax_cdf.set_xlim(0, 5); ax_cdf.set_ylim(0, 101)
        ax_cdf.legend(fontsize=6.5, ncol=2, loc="lower right")
        ax_cdf.grid(True, alpha=0.3, linestyle="--")
        ax_cdf.spines["top"].set_visible(False)
        ax_cdf.spines["right"].set_visible(False)
        plt.tight_layout()
        plt.savefig(os.path.join(REPO_ROOT, "summary_cdf.png"),
                    dpi=130, bbox_inches="tight")
        plt.show()
        print("📊 Cumulative error CDF saved → summary_cdf.png")

    # ─────────────────────────────────────────────────────────
    # FIG D — RESOURCE EFFICIENCY  (params/depth vs MAE)
    # ─────────────────────────────────────────────────────────
    _grad_rows = _ok[_ok.get("model_type", pd.Series(["gradient"]*len(_ok))) == "gradient"]                  if "model_type" in _ok.columns else _ok
    _grad_rows = _grad_rows.copy()
    try:
        _grad_rows["params_n"] = pd.to_numeric(_grad_rows["params"], errors="coerce")
        _grad_rows = _grad_rows.dropna(subset=["params_n", "test_mae"])
    except Exception:
        _grad_rows = pd.DataFrame()

    if not _grad_rows.empty and len(_grad_rows) > 1:
        fig_res, axes_res = plt.subplots(1, 2, figsize=(13, 5))

        # subplot A — params vs MAE
        ax_res0 = axes_res[0]
        sc0 = ax_res0.scatter(
            _grad_rows["params_n"], _grad_rows["test_mae"],
            s=90, c=plt.cm.viridis(np.linspace(0.15, 0.85, len(_grad_rows))),
            edgecolors="k", linewidths=0.5, zorder=3)
        for _, rr in _grad_rows.iterrows():
            ax_res0.annotate(rr["name"][:16],
                             (rr["params_n"], rr["test_mae"]),
                             fontsize=7, xytext=(4, 3),
                             textcoords="offset points")
        ax_res0.set_xlabel("Trainable Parameters", fontsize=10)
        ax_res0.set_ylabel("Test MAE (g/dL)", fontsize=10)
        ax_res0.set_title("Model Complexity vs Accuracy\n(lower-left = efficient)",
                          fontsize=10, fontweight="bold")
        ax_res0.grid(True, alpha=0.3, linestyle="--")
        ax_res0.spines["top"].set_visible(False)
        ax_res0.spines["right"].set_visible(False)

        # subplot B — training time vs MAE
        ax_res1 = axes_res[1]
        _time_rows = _ok.copy()
        _time_rows["time_n"] = pd.to_numeric(_time_rows.get("time_s"), errors="coerce")
        _time_rows = _time_rows.dropna(subset=["time_n", "test_mae"])
        if len(_time_rows) > 1:
            sc1 = ax_res1.scatter(
                _time_rows["time_n"], _time_rows["test_mae"],
                s=90, c=plt.cm.plasma(np.linspace(0.15, 0.85, len(_time_rows))),
                edgecolors="k", linewidths=0.5, zorder=3)
            for _, rr in _time_rows.iterrows():
                ax_res1.annotate(rr["name"][:16],
                                 (rr["time_n"], rr["test_mae"]),
                                 fontsize=7, xytext=(4, 3),
                                 textcoords="offset points")
            ax_res1.set_xlabel("Training Time (s)", fontsize=10)
            ax_res1.set_ylabel("Test MAE (g/dL)", fontsize=10)
            ax_res1.set_title("Training Cost vs Accuracy\n(lower-left = Pareto-optimal)",
                              fontsize=10, fontweight="bold")
            ax_res1.grid(True, alpha=0.3, linestyle="--")
            ax_res1.spines["top"].set_visible(False)
            ax_res1.spines["right"].set_visible(False)

        fig_res.suptitle("Quantum Model Resource Efficiency",
                         fontsize=13, fontweight="bold")
        plt.tight_layout()
        plt.savefig(os.path.join(REPO_ROOT, "summary_resources.png"),
                    dpi=130, bbox_inches="tight")
        plt.show()
        print("📊 Resource efficiency plot saved → summary_resources.png")

    # ─────────────────────────────────────────────────────────
    # FIG E — per-model WITHIN-TOLERANCE TABLE (paper-ready)
    # ─────────────────────────────────────────────────────────
    tols = [0.5, 1.0, 1.5, 2.0, 3.0]
    tol_data = []
    for nm in _names_all:
        if nm not in _preds:
            continue
        ae = np.abs(_preds[nm]["y_pred"] - _preds[nm]["y_true"])
        row = {"Model": nm[:35]}
        for t in tols:
            row[f"±{t}g"] = f"{(ae<=t).mean()*100:.0f}%"
        tol_data.append(row)

    if tol_data:
        df_tol = pd.DataFrame(tol_data)
        fig_tbl, ax_tbl = plt.subplots(
            figsize=(max(10, len(tols)*2.2), max(3, len(tol_data)*0.55 + 1.5)))
        ax_tbl.axis("off")
        tbl = ax_tbl.table(
            cellText=df_tol.values,
            colLabels=df_tol.columns,
            cellLoc="center", loc="center",
        )
        tbl.auto_set_font_size(False)
        tbl.set_fontsize(9)
        tbl.scale(1, 1.6)
        # Header styling
        for j in range(len(df_tol.columns)):
            tbl[(0, j)].set_facecolor("#2C3E50")
            tbl[(0, j)].set_text_props(color="white", fontweight="bold")
        # Alternate row shading
        for i in range(1, len(tol_data) + 1):
            bg = "#EAF2FB" if i % 2 == 0 else "white"
            for j in range(len(df_tol.columns)):
                tbl[(i, j)].set_facecolor(bg)
        ax_tbl.set_title("% Test Samples within Error Tolerance (g/dL)",
                         fontsize=11, fontweight="bold", pad=10)
        plt.tight_layout()
        plt.savefig(os.path.join(REPO_ROOT, "summary_tolerance_table.png"),
                    dpi=130, bbox_inches="tight")
        plt.show()
        print("📊 Tolerance table saved → summary_tolerance_table.png")

elif TASK == "regression" and not _preds:
    print("⚠  No MODEL_PREDS stored — run models first before generating paper figures.")

